In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from analysis.src.constants import OUTCOME_ORDER, PRIMARY_ADJUSTMENT
from analysis.src.io import load_data, read_table
from paths import paths

In [ ]:
OUT = paths.results.tables
OUT.directory.mkdir(parents = True, exist_ok = True)

inputs = paths.analysis.experiments

walkability_sd = load_data().walkability_score.std(ddof = 1)
scale = 100 * walkability_sd

In [ ]:
# Primary associations and environmental adjustments
primary = read_table(inputs['associations/primary'].primary_csv).set_index('outcome')
sequential = read_table(inputs['associations/primary'].sequential_models_csv)

associations = primary[['n_lsoas', 'fdr_q_value']].reindex(OUTCOME_ORDER).copy()
models = {
    'Primary': 'primary',
    'Primary + NOx': 'nox',
    'Primary + NDVI': 'ndvi',
    'Primary + NOx + NDVI': 'nox_ndvi'
}

for model, prefix in models.items():
    estimates = sequential.loc[sequential.model == model].set_index('outcome')

    for column in ['coefficient', 'ci_lower', 'ci_upper']:
        associations[f'{prefix}_{column}_pp_per_sd'] = estimates[column] * scale

    associations[f'{prefix}_p_value'] = estimates.p_value
    associations[f'{prefix}_attenuation_percent'] = estimates.attenuation_percent

associations.to_csv(OUT.primary, index_label = 'outcome')

display(associations)

In [ ]:
# Primary associations
primary_columns = {
    'primary_coefficient_pp_per_sd': primary.coefficient * scale,
    'primary_ci_lower_pp_per_sd': primary.ci_lower * scale,
    'primary_ci_upper_pp_per_sd': primary.ci_upper * scale,
    'primary_p_value': primary.p_value,
    'primary_attenuation_percent': primary.attenuation_percent
}

In [ ]:
# Within-borough comparisons
borough = read_table(inputs['spatial/borough_fixed_effects'].borough_models_csv)
borough = borough.loc[
    borough.adjustment.eq(PRIMARY_ADJUSTMENT) & borough.model.eq('Borough fixed effects')
].set_index('outcome')

borough_columns = {
    'borough_coefficient_pp_per_sd': borough.coefficient_per_sd * 100,
    'borough_ci_lower_pp_per_sd': borough.ci_lower_per_sd * 100,
    'borough_ci_upper_pp_per_sd': borough.ci_upper_per_sd * 100,
    'borough_p_value': borough.p_value,
    'borough_attenuation_percent': borough.attenuation_percent
}

In [ ]:
# Centre–periphery adjustment
centre = read_table(inputs['spatial/centre_periphery_adjustment'].centre_periphery_models_csv)
centre = centre.loc[
    centre.adjustment.eq(PRIMARY_ADJUSTMENT) & centre.model.eq('Centre–periphery adjusted')
].set_index('outcome')

centre_columns = {
    'centre_coefficient_pp_per_sd': centre.coefficient_per_sd * 100,
    'centre_ci_lower_pp_per_sd': centre.ci_lower_per_sd * 100,
    'centre_ci_upper_pp_per_sd': centre.ci_upper_per_sd * 100,
    'centre_p_value': centre.p_value,
    'centre_attenuation_percent': centre.attenuation_percent
}

In [ ]:
# Spatial-error models
spatial_error = read_table(inputs['spatial/autocorrelation_and_error_models'].spatial_error_models_csv)
spatial_error = spatial_error.loc[spatial_error.adjustment.eq(PRIMARY_ADJUSTMENT)].set_index('outcome')

relative_percent = 100 * np.expm1(spatial_error.spatial_error_coefficient * walkability_sd)

spatial_error_columns = {
    'glm_coefficient': spatial_error.glm_coefficient,
    'spatial_error_coefficient': spatial_error.spatial_error_coefficient,
    'spatial_error_relative_percent_per_sd': relative_percent,
    'spatial_error_attenuation_percent': spatial_error.spatial_error_attenuation_percent,
    'spatial_error_lambda': spatial_error.spatial_error_lambda
}

In [ ]:
# Conditional spatial null
null = read_table(inputs['spatial/conditional_null'].conditional_null_summary_csv)
null = null.loc[null.adjustment.eq(PRIMARY_ADJUSTMENT)].set_index('outcome')

null_columns = {
    'conditional_null_p_value': null.empirical_two_sided_p,
    'conditional_null_q_value': null.empirical_q_value
}

In [ ]:
# Spatial cross-validation
cv = read_table(inputs['prediction/spatial_cross_validation'].spatial_cross_validation_csv)
cv = cv.drop_duplicates(['outcome', 'fold']).groupby('outcome').incremental_r_squared.agg(['mean', 'min', 'max'])

cv_columns = {f'spatial_cv_{statistic}_incremental_r_squared': cv[statistic] for statistic in ['mean', 'min', 'max']}

In [ ]:
# Assemble spatial sensitivity table
spatial = pd.DataFrame({
    'n_lsoas': primary.n_lsoas,
    **primary_columns,
    **borough_columns,
    **centre_columns,
    **spatial_error_columns,
    **null_columns,
    **cv_columns
}).reindex(OUTCOME_ORDER)

spatial.to_csv(OUT.spatial, index_label = 'outcome')

display(spatial)